# IMDB face embedding — GPU run on vast.ai

Run these cells **in order, top to bottom**. Each one does one step. Read the short note above
each cell before running it — a couple of them need you to check the printed output before
moving to the next cell.

**Cost reminder:** this instance bills per hour while running. Once the final cell finishes and
you've downloaded the results, stop the instance from the vast.ai dashboard.

## Step 1 — Check what hardware we actually have

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import os
print("CPU cores:", os.cpu_count())
print(subprocess.run(["df", "-h", "/"], capture_output=True, text=True).stdout)

## Step 2 — Install packages

Uses `{sys.executable} -m pip` rather than plain `!pip` - on some images `!pip` installs into a
*different* Python than the notebook kernel is actually running, which causes a confusing
"module not found" later even though the install appeared to succeed. This guarantees it
installs into the same Python this notebook uses.

This installs `onnxruntime-gpu` (not the CPU-only `onnxruntime`) so the model actually runs on
the GPU. If step 1 showed no GPU, edit the pip line below to install plain `onnxruntime` instead
before running this cell.

In [ ]:
import sys
!{sys.executable} -m pip install -q numpy scipy pandas opencv-python-headless pillow insightface onnxruntime-gpu
import onnxruntime as ort
print("onnxruntime imported OK, version", ort.__version__)

## Step 3 — Confirm the GPU provider is actually available

This should print `CUDAExecutionProvider` in the list. If it doesn't, the GPU install didn't
work and everything below will silently fall back to CPU (still fine, just slower).

In [ ]:
print(ort.get_available_providers())

## Step 4 — Download the dataset

Downloads straight onto this instance's fast connection — should take well under a minute for
the ~7GB file given this instance's bandwidth.

In [ ]:
import os
os.makedirs("/workspace/imdb_job/raw", exist_ok=True)
os.chdir("/workspace/imdb_job/raw")
!wget -c https://data.vision.ee.ethz.ch/cvl/rrothe/imdb-wiki/static/imdb_crop.tar
!wget -c https://data.vision.ee.ethz.ch/cvl/rrothe/imdb-wiki/static/imdb_meta.tar
!ls -lh

## Step 5 — Extract

Using the `/workspace` volume (150GB free), so there's plenty of room to keep both the `.tar`
files and their extracted contents - no need to delete anything here.

In [ ]:
import subprocess

r1 = subprocess.run(["tar", "-xf", "imdb_crop.tar"])
r2 = subprocess.run(["tar", "-xf", "imdb_meta.tar"])
assert r1.returncode == 0 and r2.returncode == 0, "extraction failed - investigate before continuing"

!find /workspace/imdb_job/raw -maxdepth 2
!find /workspace/imdb_job/raw -iname "imdb.mat"
!df -h /workspace

## Step 6 — Build the manifest

If the `find` output above showed `imdb.mat` in a different location than assumed below,
update `MAT_PATH` to match before running.

In [ ]:
import scipy.io
import numpy as np
import pandas as pd

MAT_PATH = "/workspace/imdb_job/raw/imdb_crop/imdb.mat"  # adjust if step 5's `find` showed a different path
if not os.path.exists(MAT_PATH):
    MAT_PATH = "/workspace/imdb_job/raw/imdb_meta/imdb.mat"
print("Using:", MAT_PATH)

mat = scipy.io.loadmat(MAT_PATH)
data = mat["imdb"][0, 0]
n = data["full_path"].shape[1]

rows = []
for i in range(n):
    full_path = data["full_path"][0, i][0]
    name_arr = data["name"][0, i]
    name = name_arr[0] if name_arr.size else None
    gender = float(data["gender"][0, i])
    face_score = float(data["face_score"][0, i])
    second_face_score = float(data["second_face_score"][0, i])
    photo_taken = int(data["photo_taken"][0, i])
    rows.append({"path": full_path, "name": name, "gender": gender, "face_score": face_score,
                 "second_face_score": second_face_score, "photo_taken": photo_taken})

df = pd.DataFrame(rows)
print("Total records:", len(df))
usable = df[np.isfinite(df["face_score"])].copy()
print("Usable (finite face_score):", len(usable))
usable.to_csv("/workspace/imdb_job/imdb_manifest.csv", index=False, encoding="utf-8")
print("Wrote manifest.")

## Step 7 — Locate the actual image folder

Should print a folder full of `00`, `01`, ... `99` subfolders — that's `IMAGE_BASE` used below.

In [ ]:
!ls /workspace/imdb_job/raw/imdb_crop | head -5
IMAGE_BASE = "/workspace/imdb_job/raw/imdb_crop"  # adjust if the listing above shows a nested folder instead

## Step 8 — Define the embedding function

Same detection-confidence filtering (0.75 threshold) and chunked checkpointing approach used
locally, so the output files drop in directly alongside the existing WIKI chunks later.
Uses GPU if step 3 showed `CUDAExecutionProvider`, otherwise falls back to CPU automatically.

In [ ]:
import cv2
import time
import glob
from insightface.app import FaceAnalysis

MIN_DET_SCORE = 0.75
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPUExecutionProvider"]
print("Using providers:", providers)

app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "recognition"], providers=providers)
app.prepare(ctx_id=0, det_size=(320, 320))

def run_embedding(manifest_df, chunks_dir, save_every=1000, limit=None):
    os.makedirs(chunks_dir, exist_ok=True)
    existing = sorted(glob.glob(os.path.join(chunks_dir, "chunk_*_manifest.csv")))
    done_paths = set()
    for f in existing:
        done_paths.update(pd.read_csv(f, encoding="utf-8")["path"])
    next_idx = len(existing)

    df = manifest_df.sort_values("face_score", ascending=False)
    if limit:
        df = df.head(limit)
    df = df[~df["path"].isin(done_paths)].reset_index(drop=True)
    print(f"Already done: {len(done_paths)}, remaining this run: {len(df)}")

    batch_emb, batch_rows = [], []
    skip = {"unreadable": 0, "no_face": 0, "low_confidence": 0}
    kept_total = len(done_paths)
    t0 = time.time()

    def flush():
        nonlocal batch_emb, batch_rows, next_idx
        if not batch_emb:
            return
        emb_array = np.stack(batch_emb).astype(np.float16)
        pd.DataFrame(batch_rows).to_csv(os.path.join(chunks_dir, f"chunk_{next_idx:05d}_manifest.csv"),
                                        index=False, encoding="utf-8")
        np.save(os.path.join(chunks_dir, f"chunk_{next_idx:05d}_emb.npy"), emb_array)
        next_idx += 1
        batch_emb, batch_rows = [], []

    for i, row in df.iterrows():
        img = cv2.imread(os.path.join(IMAGE_BASE, row["path"]))
        if img is None:
            skip["unreadable"] += 1
            continue
        faces = app.get(img)
        if not faces:
            skip["no_face"] += 1
            continue
        face = max(faces, key=lambda f: f.det_score)
        if face.det_score < MIN_DET_SCORE:
            skip["low_confidence"] += 1
            continue
        batch_emb.append(face.normed_embedding)
        batch_rows.append(row)
        kept_total += 1
        if len(batch_emb) >= save_every:
            flush()
            elapsed = time.time() - t0
            print(f"{i+1}/{len(df)} processed, {elapsed:.1f}s elapsed, {elapsed/(i+1):.4f}s/img, "
                  f"kept_total={kept_total}")
    flush()
    print(f"Done. skipped={skip}, kept_total={kept_total}")

manifest_df = pd.read_csv("/workspace/imdb_job/imdb_manifest.csv", encoding="utf-8")

## Step 9 — Quick test (500 images)

Check the `s/img` figure this prints. If it's not dramatically faster than ~0.5s/img (the CPU
laptop baseline), the GPU probably isn't actually being used — check step 3's output again
before running the full job.

In [ ]:
run_embedding(manifest_df, "/workspace/imdb_job/test_chunks", save_every=100, limit=500)

## Step 10 — The full run (all usable IMDB images)

In [ ]:
run_embedding(manifest_df, "/workspace/imdb_job/imdb_chunks", save_every=1000)

## Step 11 — Zip the results for download

Download the resulting zip via Jupyter's file browser (left sidebar → navigate to
`imdb_job/imdb_chunks.zip` → right-click → Download).

In [ ]:
import shutil
shutil.make_archive("/workspace/imdb_job/imdb_chunks", "zip", "/workspace/imdb_job/imdb_chunks")
print(os.path.getsize("/workspace/imdb_job/imdb_chunks.zip") / 1e6, "MB")

## Done

Once you've downloaded `imdb_chunks.zip`, **stop this instance from the vast.ai dashboard** to
stop the hourly billing.